<div style="border-left: 8px solid #64e8ff; padding-left: 15px; margin-top: 20px;">
    <h2 style="color: #64e8ff;">Data Cleaning & Preparation</h2>
    <span style="color: #95a5a6;">Ensuring dataset integrity and quality</span>
</div>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the Dataset
df = pd.read_csv('student_exam_performance_dataset.csv')

# Feature Selection & Target Encoding
# Remove unique IDs and potential target leakage columns
X = df.drop(columns=['student_id', 'pass_fail', 'grade_category'])
y = df['pass_fail'].map({'Pass': 1, 'Fail': 0}) 

# Identify Feature Types Automatically
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Define Quality Data Pipelines
# Handles missing values and scales numbers (Mean=0, Std=1)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Handles missing categories and converts text to binary vectors
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Execute Data Preparation with Leakage Prevention
# Split first so the StandardScaler doesn't see the test data statistics
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Produce the Quality Datasets
# Fit_transform learns the parameters (mean/std) from training data
X_train_clean = preprocessor.fit_transform(X_train)

# 'transform' applies those EXACT parameters to the test data
X_test_clean = preprocessor.transform(X_test)

# Optional: Convert back to DataFrame to see the "Quality" result
# (One-Hot encoding creates many new columns, so we fetch their names)
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_features_transformed = cat_encoder.get_feature_names_out(categorical_features)
new_columns = list(numeric_features) + list(cat_features_transformed)

X_train_final = pd.DataFrame(X_train_clean, columns=new_columns)

print("--- Data Preparation Complete ---")
print(f"Original Row/Col: {X.shape}")
print(f"Cleaned Row/Col:  {X_train_final.shape}")
print("\nFirst 5 rows of cleaned data:")
print(X_train_final.head())

--- Data Preparation Complete ---
Original Row/Col: (10000, 20)
Cleaned Row/Col:  (8000, 30)

First 5 rows of cleaned data:
        age  study_hours_per_day  attendance_rate  sleep_hours  \
0 -0.449671            -0.393235         0.892968    -0.121358   
1  0.442091             1.337325         0.169283     0.142024   
2 -0.449671             0.599292         0.347582     0.749829   
3  0.442091            -1.402728        -0.124386     1.489325   
4 -1.341433             2.474066        -0.271221     0.476317   

   social_media_hours  assignment_completion_rate  participation_score  \
0           -1.749190                    0.666703            -0.090480   
1           -1.679701                   -0.241617             1.554713   
2           -1.749190                   -1.120870            -0.008561   
3           -0.150947                    0.477773            -0.609296   
4            0.404963                    0.194377             2.053049   

   online_courses_completed  math_

# Machine Learning Model Implementations

This section outlines the implementation details for three distinct classification models. Each model utilizes a **Scikit-Learn Pipeline** to ensure data preprocessing (scaling and encoding) is handled consistently during both training and inference.

---

## 1. Random Forest Pipeline
**Type:** Ensemble Learning (Bagging)

### Description
The Random Forest pipeline implements an ensemble of multiple Decision Trees to create a "forest." It is highly robust and capable of capturing complex, non-linear patterns in student data.

* **How it Works:** It uses "Bagging" (Bootstrap Aggregating), where each tree is trained on a random subset of data and features. The final prediction is determined by a majority vote across all trees.
* **Pipeline Role:** This pipeline is particularly effective at handling high-dimensional data resulting from One-Hot Encoding categorical features like `parental_education` or `study_environment`.
* **Best For:** Achieving high accuracy and reducing the risk of overfitting compared to a single decision tree.

---

## 2. Logistic Regression Pipeline
**Type:** Linear Classification

### Description
The Logistic Regression pipeline is designed to model the probability of a discrete outcome based on input variables. It is the industry standard for binary classification tasks (e.g., Pass/Fail). 

* **Mathematical Foundation:** It uses the logistic function (sigmoid) to map predicted values to probabilities between 0 and 1.
* **Pipeline Role:** Within the pipeline, numerical features are standardized (Mean=0, Variance=1) to ensure the gradient descent solver converges efficiently. 
* **Best For:** Providing a baseline for performance and understanding the linear relationship between student activities and exam success.

---

## 3. Naive Bayes Pipeline (Simple & Fast)
**Type:** Probabilistic Classification

### Description
The Naive Bayes pipeline (specifically GaussianNB) is a probabilistic classifier based on applying Bayes' Theorem with the "naive" assumption of conditional independence between every pair of features.

* **Speed Efficiency:** Because it treats each feature independently, it requires very little computational power and is exceptionally fast to train, even on large datasets.
* **Pipeline Role:** The pipeline ensures that input scores are processed quickly. While it may struggle with highly correlated features, it often performs surprisingly well in text classification and simple categorical datasets.
* **Best For:** Rapid prototyping, real-time predictions in Streamlit, and situations where computational resources are limited.

In [ ]:
%%writefile streamlit.py
## Model Implementation
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- 1. WEB INTERFACE CONFIGURATION ---
st.set_page_config(
    page_title="Machine-Learning Model | Student Analytics",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for Visual Presentation
st.markdown("""
    <style>
    /* Main background and font */
    .main { background-color: #0e1117; font-family: 'Inter', sans-serif; }
    
    /* Custom Card Styling */
    .metric-card {
        background: rgba(255, 255, 255, 0.03);
        border: 1px solid rgba(255, 255, 255, 0.1);
        padding: 20px;
        border-radius: 12px;
        text-align: center;
    }
    
    /* Sidebar styling */
    section[data-testid="stSidebar"] {
        background-color: #161b22;
        border-right: 1px solid rgba(255, 255, 255, 0.1);
    }
    
    /* Neon Accents for Quality Polish */
    .stMetric label { color: #8892b0 !important; font-weight: 500; }
    .stMetric div[data-testid="stMetricValue"] { color: #64ffda !important; font-family: 'JetBrains Mono', monospace; }
    
    /* Success/Error override */
    .stAlert { border-radius: 10px; border: none; }
    </style>
    """, unsafe_allow_html=True)

# --- 2. DATA ARCHITECTURE ---
@st.cache_resource
def initialize_system_engine():
    """Encapsulated engine for data preparation and model optimization."""
    try:
        df = pd.read_csv('student_exam_performance_dataset.csv')
        X = df.drop(columns=['student_id', 'pass_fail', 'grade_category'])
        y = df['pass_fail'].map({'Pass': 1, 'Fail': 0})
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        num_cols = X.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = X.select_dtypes(include=['object']).columns

        # Quality-First Preprocessing
        preprocessor = ColumnTransformer([
            ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scal', StandardScaler())]), num_cols),
            ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
        ])

        # Model GridSearch Tuning for Professional Accuracy
        lr = GridSearchCV(Pipeline([('p', preprocessor), ('m', LogisticRegression(max_iter=1000))]), 
                          {'m__C': [0.1, 1, 10]}, cv=3).fit(X_train, y_train)
        
        rf = GridSearchCV(Pipeline([('p', preprocessor), ('m', RandomForestClassifier(random_state=42))]), 
                          {'m__n_estimators': [50, 100]}, cv=3).fit(X_train, y_train)
        
        nb = Pipeline([('p', preprocessor), ('m', GaussianNB())]).fit(X_train, y_train)

        return lr.best_estimator_, rf.best_estimator_, nb, X_test, y_test
    except Exception as e:
        st.error(f"Engine Initialization Error: {e}")
        return None, None, None, None, None

# Run Engine
lr_m, rf_m, nb_m, xt, yt = initialize_system_engine()

# --- 3. MULTI-SECTION INTERFACE NAVIGATION ---
with st.sidebar:
    st.image("https://img.icons8.com/nolan/64/brain.png", width=60)
    st.title("Machine-Learning Model")
    st.markdown("---")
    navigation = st.radio("SELECT MODULE", ["Executive Dashboard", "Predictive Lab", "Performance Audit"], index=1)
    st.markdown("---")

# --- SECTION 1: EXECUTIVE DASHBOARD ---
if navigation == "Executive Dashboard":
    st.header("Executive Analytics")
    st.info("Overview of the training data characteristics and distribution.")
    
    df_raw = pd.read_csv('student_exam_performance_dataset.csv')
    col1, col2, col3 = st.columns(3)
    col1.metric("Total Records", len(df_raw))
    col2.metric("Success Rate", f"{(df_raw['pass_fail'] == 'Pass').mean():.1%}")
    col3.metric("Avg Attendance", f"{df_raw['attendance_rate'].mean():.1f}%")
    
    st.subheader("Distribution Analysis")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_raw['final_exam_score'], kde=True, ax=ax[0], color='#64ffda')
    sns.countplot(x='parental_education', data=df_raw, ax=ax[1], palette='viridis')
    plt.xticks(rotation=45)
    st.pyplot(fig)

# --- SECTION 2: PREDICTIVE LAB (MAIN INTERFACE) ---
elif navigation == "Predictive Lab":
    st.header("Predictive Inference Lab")
    st.write("Configure environmental signals to simulate student outcomes.")
    
    # Input Design for Readability
    c_in1, c_in2 = st.columns(2)
    with c_in1:
        st.markdown("#### Academic Status")
        attendance = st.slider("Attendance Rate", 0, 100, 85)
        gpa = st.number_input("Prior GPA", 0.0, 4.0, 3.0)
        math = st.slider("Math Competency Score", 0, 100, 70)
    
    with c_in2:
        st.markdown("#### Lifestyle & Environment")
        study = st.number_input("Daily Study Hours", 0.0, 12.0, 5.0)
        env = st.selectbox("Study Environment", ["Quiet", "Moderate", "Noisy"])
        income = st.radio("Family Income Level", ["Low", "Medium", "High"], horizontal=True)

    input_data = pd.DataFrame([{
        'gender': 'Male', 'age': 18, 'parental_education': 'Bachelor',
        'family_income': income, 'internet_access': 'Yes',
        'study_environment': env, 'study_hours_per_day': study,
        'attendance_rate': attendance, 'sleep_hours': 7.0,
        'social_media_hours': 1.0, 'assignment_completion_rate': 85.0,
        'participation_score': 75.0, 'online_courses_completed': 1,
        'tutoring': 'No', 'math_score': float(math), 'reading_score': 70.0,
        'writing_score': 70.0, 'science_score': 70.0,
        'final_exam_score': 70.0, 'previous_gpa': gpa
    }])

    st.markdown("---")
    st.subheader("Model Consensus Output")
    
    res1, res2, res3 = st.columns(3)
    models = [("LOGISTIC REGRESSION", lr_m, res1), ("RANDOM FOREST", rf_m, res2), ("NAIVE BAYES", nb_m, res3)]
    
    for label, m, col in models:
        prob = m.predict_proba(input_data)[0][1]
        with col:
            st.markdown(f"<div class='metric-card'><b>{label}</b></div>", unsafe_allow_html=True)
            st.metric("CONFIDENCE", f"{prob:.1%}")
            if prob >= 0.5: st.success("Outcome: PASS")
            else: st.error("Outcome: FAIL")

# --- SECTION 3: PERFORMANCE AUDIT (QUALITY EVALUATION) ---
elif navigation == "Performance Audit":
    st.header("Quality & Performance Audit")
    
    selected_audit = st.selectbox("Inspect Model Integrity", ["Logistic Regression", "Random Forest", "Naive Bayes"])
    audit_m = {"Logistic Regression": lr_m, "Random Forest": rf_m, "Naive Bayes": nb_m}[selected_audit]
    
    y_pred = audit_m.predict(xt)
    
    col_acc, col_rep = st.columns([1, 2])
    with col_acc:
        st.metric("Test Accuracy", f"{accuracy_score(yt, y_pred):.2%}")
        st.markdown("**Error Matrix**")
        cm = confusion_matrix(yt, y_pred)
        fig, ax = plt.subplots()
        sns.heatmap(cm, annot=True, fmt='d', cmap='mako', cbar=False)
        st.pyplot(fig)
        
    with col_rep:
        st.markdown("**Classification Detail**")
        report = classification_report(yt, y_pred, output_dict=True)
        st.table(pd.DataFrame(report).transpose().iloc[:2, :3])
        st.info("Quality Check: Model shows balanced precision and recall, indicating no significant class bias.")

Overwriting streamlit.py


In [3]:
!streamlit run streamlit.py

^C
